In [1]:
"""
Main Engine (Clean, Single-File)
- Dataset: MHEALTH (.log)
- LOSO per-activity
- Detector: block-level c_full -> robust_z -> steady score s_full, window slices
- Fusion: self-consistency quality -> softmax weights
- Model: latent z + transition prob p_hat(t) + reconstruction
- Training: recon + soft pseudo supervision + pair consistency + inertial invariance
- Outputs:
  - results_loso.csv
  - 5 plots (per activity representative fold + act summary)

+ ✅ 3-stage hierarchical sweep (NO OTHER LOGIC CHANGED)
(1) quality_tau candidates
(2) best quality_tau + ref_q candidates
(3) best (quality_tau, ref_q) + sigmoid_tau candidates
총 9~15 runs
"""

import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.signal import savgol_filter
import matplotlib.pyplot as plt

# =========================================================
# 0) CONFIG
# =========================================================
CONFIG = {
    # data
    "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
    "target_activities": [6, 7, 12],
    "fs": 50,

    # windowing
    "window_size": 100,
    "stride": 50,

    # groups (missing-friendly)
    "groups": ["chest_acc", "ankle_acc", "arm_acc", "ankle_gyro", "arm_gyro"],

    # training
    "batch_size": 64,
    "epochs": 20,
    "lr": 1e-3,
    "seed": 42,

    # model
    "latent_dim": 64,
    "hidden_dim": 128,

    # losses
    "lambda_recon": 1.0,
    "lambda_soft": 0.5,
    "lambda_pair": 0.3,
    "lambda_inertial": 0.2,

    # detector smoothing
    "det_savgol": True,
    "det_savgol_win": 11,
    "det_savgol_poly": 2,

    # detector robust normalization
    "mad_eps": 1e-6,
    "sigmoid_tau": 1.0,
    "sigmoid_ref_q": 80,
    "mad_floor": 1e-3,

    # fusion quality (self-consistency)
    "quality_quantile": 0.2,
    "quality_pairs": 128,
    "quality_tau": 1.0,

    # pair loss / report
    "pair_delta": 5,
    "pair_margin": 0.2,
    "pair_q": 0.2,  # top/bottom quantile for SS/TT/ST/TS in report

    # output
    "out_dir": "./out_main_engine",
    "plot_sec": 60,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# =========================================================
# 1) MHEALTH LOADING
# =========================================================
def load_mhealth_df(data_dir: str, target_activities):
    log_files = glob.glob(os.path.join(data_dir, "*.log"))
    if not log_files:
        raise FileNotFoundError(f"No .log files found in {data_dir}")

    all_rows = []
    for fpath in log_files:
        filename = os.path.basename(fpath)
        try:
            sub_id = int("".join(filter(str.isdigit, filename)))
        except:
            sub_id = 0

        try:
            df = pd.read_csv(fpath, sep=r"\s+", header=None, engine="python")
        except:
            df = pd.read_csv(fpath, sep="\t", header=None)

        if df.shape[1] <= 23:
            raise ValueError("Unexpected MHEALTH column count.")

        df = df.copy()
        df["label"] = df.iloc[:, 23].astype(int)
        df["subject_id"] = sub_id

        df = df[df["label"].isin(target_activities)].copy()
        if len(df) > 0:
            all_rows.append(df)

    if not all_rows:
        raise ValueError("No data found for specified activities.")

    df_all = pd.concat(all_rows, ignore_index=True)
    print(f"[Data] Total samples: {len(df_all)} | Subjects: {sorted(df_all['subject_id'].unique())} | Acts: {sorted(df_all['label'].unique())}")
    return df_all


def get_group_array_from_block(block_np: np.ndarray, group: str):
    # MHEALTH indices (0-based):
    # 0-2 chest acc
    # 5-7 ankle acc
    # 8-10 ankle gyro
    # 14-16 arm acc
    # 17-19 arm gyro
    if group == "chest_acc":
        return block_np[:, 0:3]
    if group == "ankle_acc":
        return block_np[:, 5:8]
    if group == "arm_acc":
        return block_np[:, 14:17]
    if group == "ankle_gyro":
        return block_np[:, 8:11]
    if group == "arm_gyro":
        return block_np[:, 17:20]
    return None


def create_blocks_by_subject_activity(df_all: pd.DataFrame, win_size: int):
    blocks = []
    base_cols = list(range(24))
    for sub in sorted(df_all["subject_id"].unique()):
        sub_df = df_all[df_all["subject_id"] == sub]
        for act in sorted(sub_df["label"].unique()):
            act_df = sub_df[sub_df["label"] == act]
            raw = act_df[base_cols].to_numpy(dtype=np.float32)
            if len(raw) >= win_size:
                blocks.append({"subject": int(sub), "act": int(act), "raw": raw})
    return blocks


# =========================================================
# 2) Detector + Fusion
# =========================================================
def robust_zscore(x: np.ndarray, eps=1e-6, mad_floor=0.0):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + eps
    if mad_floor > 0:
        mad = max(mad, mad_floor)
    return (x - med) / mad

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def acc_change_score(acc_3: np.ndarray, var_win: int = 10):
    a = acc_3
    T = len(a)
    u = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)

    cos_prev = np.sum(u[1:] * u[:-1], axis=1)
    cos_prev = np.clip(cos_prev, -1.0, 1.0)

    dir_change = np.zeros(T, dtype=np.float32)
    dir_change[1:] = 1.0 - cos_prev

    half = var_win // 2
    dir_var = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        local = u[s:e]
        dir_var[i] = float(np.mean(np.var(local, axis=0)))

    c = dir_change + dir_var
    return c.astype(np.float32)

def gyro_change_score(gyro_3: np.ndarray, var_win: int = 10):
    w = gyro_3
    T = len(w)
    mag = np.linalg.norm(w, axis=1)

    dmag = np.zeros(T, dtype=np.float32)
    dmag[1:] = np.abs(mag[1:] - mag[:-1])

    half = var_win // 2
    lvar = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        lvar[i] = float(np.var(mag[s:e]))

    c = dmag + lvar
    return c.astype(np.float32)

def compute_group_change(group_key: str, Xg: np.ndarray, cfg):
    if Xg is None:
        return None
    if "acc" in group_key:
        return acc_change_score(Xg, var_win=10)
    if "gyro" in group_key:
        return gyro_change_score(Xg, var_win=10)
    return np.linalg.norm(np.diff(Xg, axis=0, prepend=Xg[:1]), axis=1).astype(np.float32)

def steady_score_from_change(c: np.ndarray, cfg):
    cz = robust_zscore(c, eps=cfg["mad_eps"], mad_floor=cfg["mad_floor"])
    b = np.percentile(cz, cfg["sigmoid_ref_q"])
    y = sigmoid((cz - b) / cfg["sigmoid_tau"])  # transition prob
    s = 1.0 - y                                 # steady score

    if cfg["det_savgol"] and len(s) >= cfg["det_savgol_win"] and (cfg["det_savgol_win"] % 2 == 1):
        s = savgol_filter(s, window_length=cfg["det_savgol_win"], polyorder=cfg["det_savgol_poly"]).astype(np.float32)
        s = np.clip(s, 0.0, 1.0)
    return s.astype(np.float32)

def cosine_sim(a: np.ndarray, b: np.ndarray, eps=1e-8):
    na = np.linalg.norm(a) + eps
    nb = np.linalg.norm(b) + eps
    return float(np.dot(a, b) / (na * nb))

def estimate_self_consistency_quality(group_key: str, Xw: np.ndarray, sw: np.ndarray, cfg):
    if Xw is None or sw is None:
        return 0.0
    T = len(sw)
    q = cfg["quality_quantile"]
    k_pairs = cfg["quality_pairs"]

    order = np.argsort(sw)
    n = max(5, int(q * T))
    low_idx = order[:n]
    high_idx = order[-n:]
    if len(low_idx) < 5 or len(high_idx) < 5:
        return 0.0

    if "acc" in group_key:
        V = Xw / (np.linalg.norm(Xw, axis=1, keepdims=True) + 1e-8)
    else:
        V = Xw

    def mean_pair_sim(idxs):
        sims = []
        for _ in range(k_pairs):
            i, j = np.random.choice(idxs, size=2, replace=True)
            sims.append(cosine_sim(V[i], V[j]))
        return float(np.mean(sims))

    S_high = mean_pair_sim(high_idx)
    S_low = mean_pair_sim(low_idx)
    return float(S_high - S_low)

def fuse_group_scores(groups_dict, cfg):
    avail = [(g, d) for g, d in groups_dict.items() if d["s"] is not None]
    if not avail:
        raise ValueError("No available groups for fusion.")

    qs = np.array([d["q"] for _, d in avail], dtype=np.float32)
    tau = max(cfg["quality_tau"], 1e-6)
    ws = np.exp(qs / tau)
    ws = ws / (ws.sum() + 1e-8)

    T = len(avail[0][1]["s"])
    s_fused = np.zeros(T, dtype=np.float32)
    weights, qualities = {}, {}
    for (g, d), w in zip(avail, ws):
        s_fused += float(w) * d["s"]
        weights[g] = float(w)
        qualities[g] = float(d["q"])

    s_fused = np.clip(s_fused, 0.0, 1.0)
    return s_fused, weights, qualities


# =========================================================
# 3) Dataset (block-level s_full -> window slice)
# =========================================================
class MainEngineDataset(Dataset):
    def __init__(self, blocks, cfg):
        self.cfg = cfg
        self.groups = cfg["groups"]
        self.win = cfg["window_size"]
        self.stride = cfg["stride"]

        self.samples = []  # {"x":(win,C),"s":(win,),"y":(win,), "meta":{...}}

        for b in blocks:
            raw = b["raw"]
            subject = b["subject"]
            act = b["act"]
            T = len(raw)

            full_groups = {}
            for g in self.groups:
                Xg = get_group_array_from_block(raw, g)
                full_groups[g] = None if Xg is None else Xg.astype(np.float32)

            block_cache = {}
            for g in self.groups:
                Xg_full = full_groups.get(g, None)
                if Xg_full is None:
                    block_cache[g] = {"X_full": None, "s_full": None}
                    continue
                c_full = compute_group_change(g, Xg_full, cfg)
                s_full = steady_score_from_change(c_full, cfg)  # (T,)
                block_cache[g] = {"X_full": Xg_full, "s_full": s_full}

            groups_dict_full = {}
            for g in self.groups:
                Xg_full = block_cache[g]["X_full"]
                s_full = block_cache[g]["s_full"]
                if Xg_full is None or s_full is None:
                    continue
                q_full = estimate_self_consistency_quality(g, Xg_full, s_full, cfg)
                groups_dict_full[g] = {"X": Xg_full, "s": s_full, "q": float(q_full)}

            if len(groups_dict_full) == 0:
                continue

            s_fused_full, w_block, q_block = fuse_group_scores(groups_dict_full, cfg)  # (T,)
            y_fused_full = (1.0 - s_fused_full).astype(np.float32)

            for st in range(0, T - self.win + 1, self.stride):
                ed = st + self.win

                # model input
                x_parts = []
                for g in self.groups:
                    Xg_full_in = full_groups.get(g, None)
                    if Xg_full_in is not None:
                        x_parts.append(Xg_full_in[st:ed])  # (win,3)
                if len(x_parts) == 0:
                    continue

                # slice only
                s_fused = s_fused_full[st:ed].astype(np.float32)
                y_pseudo = y_fused_full[st:ed].astype(np.float32)

                Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)  # (win,C)
                mu = Xcat.mean(axis=0, keepdims=True)
                sd = Xcat.std(axis=0, keepdims=True) + 1e-6
                Xcat = (Xcat - mu) / sd

                self.samples.append({
                    "x": Xcat,
                    "s": s_fused,
                    "y": y_pseudo,
                    # meta: now fixed per block (trial)
                    "meta": {"subject": subject, "act": act, "weights": w_block, "qualities": q_block},
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = self.samples[idx]
        x = torch.tensor(d["x"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        s = torch.tensor(d["s"], dtype=torch.float32)                  # (T,)
        y = torch.tensor(d["y"], dtype=torch.float32)                  # (T,)
        return x, s, y


# =========================================================
# 4) Model
# =========================================================
class MainEngineNet(nn.Module):
    def __init__(self, input_ch: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(input_ch, hidden_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.to_latent = nn.Sequential(
            nn.Conv1d(hidden_dim, latent_dim, kernel_size=1),
            nn.ReLU(),
        )
        self.trans_head = nn.Sequential(
            nn.Conv1d(latent_dim, 32, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(32, 1, kernel_size=1),
        )
        self.grav_proj = nn.Linear(latent_dim, 3)
        self.decoder = nn.Sequential(
            nn.Conv1d(latent_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, input_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.to_latent(h)                  # (B,D,T)
        p = torch.sigmoid(self.trans_head(z))  # (B,1,T)
        x_recon = self.decoder(z)              # (B,C,T)
        return z, p, x_recon


# =========================================================
# 5) Losses
# =========================================================
def weighted_soft_bce(p_hat, y_soft, eps=1e-6):
    p = p_hat.squeeze(1)
    y = y_soft
    return F.binary_cross_entropy(p.clamp(eps, 1 - eps), y.clamp(eps, 1 - eps))

def pair_consistency_loss(z, s, cfg):
    B, D, T = z.shape
    delta = cfg["pair_delta"]
    margin = cfg["pair_margin"]
    if T <= delta:
        return torch.tensor(0.0, device=z.device)

    z1 = z[:, :, :-delta]
    z2 = z[:, :, delta:]
    s1 = s[:, :-delta]
    s2 = s[:, delta:]

    z1n = F.normalize(z1, dim=1)
    z2n = F.normalize(z2, dim=1)
    sim = (z1n * z2n).sum(dim=1)  # (B,T-d)

    w_pos = (s1 * s2).detach()
    w_neg = ((1 - s1) * (1 - s2)).detach()

    pos_loss = (w_pos * (1.0 - sim)).mean()
    neg_loss = (w_neg * F.relu(sim - margin)).mean()
    return pos_loss + neg_loss

def inertial_invariance_loss(z, x, s, model, eps=1e-6):
    """
    Very light version: correlate projected z_bar with mean gravity direction in x (steady-weighted)
    """
    B, D, T = z.shape
    C = x.shape[1]
    nvec = C // 3
    if nvec == 0:
        return torch.tensor(0.0, device=z.device)

    x_reshaped = x[:, :nvec * 3, :].reshape(B, nvec, 3, T)
    g = x_reshaped.mean(dim=3).mean(dim=1)  # (B,3)
    g = F.normalize(g, dim=1)

    w = s / (s.sum(dim=1, keepdim=True) + eps)
    z_bar = (z * w.unsqueeze(1)).sum(dim=2)  # (B,D)

    z3 = model.grav_proj(z_bar)
    z3 = F.normalize(z3, dim=1)
    corr = torch.abs(F.cosine_similarity(z3, g, dim=1)).mean()
    return corr


# =========================================================
# 6) Train / Eval
# =========================================================
def train_one_fold(model, loader, cfg):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    model.train()
    for ep in range(cfg["epochs"]):
        L = []
        for x, s, y in loader:
            x = x.to(DEVICE)
            s = s.to(DEVICE)
            y = y.to(DEVICE)

            z, p, x_recon = model(x)

            loss_recon = F.mse_loss(x_recon, x)
            loss_soft = weighted_soft_bce(p, y)
            loss_pair = pair_consistency_loss(z, s, cfg)
            loss_inert = inertial_invariance_loss(z, x, s, model)

            loss = (cfg["lambda_recon"] * loss_recon +
                    cfg["lambda_soft"] * loss_soft +
                    cfg["lambda_pair"] * loss_pair +
                    cfg["lambda_inertial"] * loss_inert)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            L.append([loss.item(), loss_recon.item(), loss_soft.item(), loss_pair.item(), loss_inert.item()])

        sched.step()

    arr = np.array(L)
    return {
        "loss": float(arr[:,0].mean()),
        "recon": float(arr[:,1].mean()),
        "soft": float(arr[:,2].mean()),
        "pair": float(arr[:,3].mean()),
        "inert": float(arr[:,4].mean()),
    }

@torch.no_grad()
def eval_soft_mae_mse(model, loader):
    model.eval()
    all_p, all_y = [], []
    for x, s, y in loader:
        x = x.to(DEVICE)
        _, p, _ = model(x)
        all_p.append(p.squeeze(1).cpu().numpy())
        all_y.append(y.cpu().numpy())
    P = np.concatenate(all_p, axis=0)
    Y = np.concatenate(all_y, axis=0)
    mae = float(np.mean(np.abs(P - Y)))
    mse = float(np.mean((P - Y) ** 2))
    return mae, mse


# =========================================================
# 7) Reporting helpers (plots/stat) - minimal
# =========================================================
def build_full_groups_from_raw(raw_24: np.ndarray, cfg):
    full_groups = {}
    for g in cfg["groups"]:
        Xg = get_group_array_from_block(raw_24, g)
        full_groups[g] = None if Xg is None else Xg.astype(np.float32)
    return full_groups

def detector_fused_on_block(raw_24: np.ndarray, cfg):
    full_groups = build_full_groups_from_raw(raw_24, cfg)

    groups_dict = {}
    for g in cfg["groups"]:
        Xg = full_groups.get(g, None)
        if Xg is None:
            continue
        c = compute_group_change(g, Xg, cfg)
        s = steady_score_from_change(c, cfg)
        q = estimate_self_consistency_quality(g, Xg, s, cfg)
        groups_dict[g] = {"X": Xg, "s": s, "q": float(q)}

    s_fused, w_dict, q_dict = fuse_group_scores(groups_dict, cfg)
    y = (1.0 - s_fused).astype(np.float32)
    return s_fused, y, w_dict, q_dict

@torch.no_grad()
def infer_p_hat_on_block(model, raw_24: np.ndarray, cfg):
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    T = len(raw_24)

    full_groups = build_full_groups_from_raw(raw_24, cfg)
    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None

    p_sum = np.zeros(T, dtype=np.float32)
    p_cnt = np.zeros(T, dtype=np.float32)

    for st in range(0, T - win + 1, stride):
        ed = st + win
        x_parts = [full_groups[g][st:ed] for g in avail_groups]
        Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)

        mu = Xcat.mean(axis=0, keepdims=True)
        sd = Xcat.std(axis=0, keepdims=True) + 1e-6
        Xcat = (Xcat - mu) / sd

        x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
        _, p, _ = model(x)
        pw = p.squeeze(0).squeeze(0).detach().cpu().numpy().astype(np.float32)

        p_sum[st:ed] += pw
        p_cnt[st:ed] += 1.0

    return p_sum / (p_cnt + 1e-8)

def compute_pair_separation_stats(model, raw_24: np.ndarray, cfg):
    """
    Returns dict of arrays: SS/TT/ST/TS cosine sim between z_t and z_{t+delta}
    computed on overlap-avg z over time.
    """
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    delta = cfg["pair_delta"]

    # detector
    s_fused, y, _, _ = detector_fused_on_block(raw_24, cfg)

    full_groups = build_full_groups_from_raw(raw_24, cfg)
    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None

    T = len(raw_24)
    z_sum = None
    z_cnt = np.zeros(T, dtype=np.float32)

    with torch.no_grad():
        for st in range(0, T - win + 1, stride):
            ed = st + win
            x_parts = [full_groups[g][st:ed] for g in avail_groups]
            Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)
            mu = Xcat.mean(axis=0, keepdims=True)
            sd = Xcat.std(axis=0, keepdims=True) + 1e-6
            Xcat = (Xcat - mu) / sd

            x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
            z, _, _ = model(x)  # (1,D,win)
            z = z.squeeze(0).detach().cpu().numpy()  # (D,win)

            if z_sum is None:
                z_sum = np.zeros((z.shape[0], T), dtype=np.float32)

            z_sum[:, st:ed] += z
            z_cnt[st:ed] += 1.0

    z_full = z_sum / (z_cnt[None, :] + 1e-8)  # (D,T)

    if T <= delta + 1:
        return None

    z1 = z_full[:, :-delta]
    z2 = z_full[:, delta:]
    z1n = z1 / (np.linalg.norm(z1, axis=0, keepdims=True) + 1e-8)
    z2n = z2 / (np.linalg.norm(z2, axis=0, keepdims=True) + 1e-8)
    sim = np.sum(z1n * z2n, axis=0)  # (T-delta,)

    y1 = y[:-delta]
    y2 = y[delta:]

    q = float(cfg.get("pair_q", 0.2))
    thr_hi = float(np.quantile(y, 1.0 - q))
    thr_lo = float(np.quantile(y, q))

    mask_ss = (y1 <= thr_lo) & (y2 <= thr_lo)
    mask_tt = (y1 >= thr_hi) & (y2 >= thr_hi)
    mask_st = (y1 <= thr_lo) & (y2 >= thr_hi)
    mask_ts = (y1 >= thr_hi) & (y2 <= thr_lo)

    return {
        "thr_lo": thr_lo,
        "thr_hi": thr_hi,
        "SS": sim[mask_ss],
        "TT": sim[mask_tt],
        "ST": sim[mask_st],
        "TS": sim[mask_ts],
    }

def plot_p1_raw_and_y(raw_24, y, cfg, title, save_path):
    fs = cfg["fs"]
    T_show = min(int(cfg["plot_sec"] * fs), len(raw_24))
    t = np.arange(T_show) / fs

    def mag(g):
        Xg = get_group_array_from_block(raw_24[:T_show], g)
        if Xg is None:
            return None
        return np.linalg.norm(Xg, axis=1)

    chest_acc = mag("chest_acc")
    ankle_acc = mag("ankle_acc")
    arm_acc = mag("arm_acc")
    ankle_gyro = mag("ankle_gyro")
    arm_gyro = mag("arm_gyro")

    plt.figure(figsize=(14, 7))
    ax1 = plt.subplot(2, 1, 1)
    if chest_acc is not None: ax1.plot(t, chest_acc, label="chest_acc |mag|", alpha=0.9)
    if ankle_acc is not None: ax1.plot(t, ankle_acc, label="ankle_acc |mag|", alpha=0.9)
    if arm_acc is not None: ax1.plot(t, arm_acc, label="arm_acc |mag|", alpha=0.9)
    if ankle_gyro is not None: ax1.plot(t, ankle_gyro, label="ankle_gyro |mag|", alpha=0.7)
    if arm_gyro is not None: ax1.plot(t, arm_gyro, label="arm_gyro |mag|", alpha=0.7)
    ax1.set_title(title + " | raw magnitudes")
    ax1.grid(alpha=0.25)
    ax1.legend(ncol=3)

    ax2 = plt.subplot(2, 1, 2)
    ax2.plot(t, y[:T_show], label="pseudo y = transition (1 - s_fused)")
    ax2.set_ylim(0, 1)
    ax2.grid(alpha=0.25)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

def plot_p2_y_vs_phat(y, p_hat, cfg, title, save_path):
    fs = cfg["fs"]
    T_show = min(int(cfg["plot_sec"] * fs), len(y))
    t = np.arange(T_show) / fs

    plt.figure(figsize=(14, 4))
    plt.plot(t, y[:T_show], label="pseudo y", alpha=0.7)
    if p_hat is not None:
        plt.plot(t, p_hat[:T_show], label="model p_hat", alpha=0.9)
    plt.ylim(0, 1)
    plt.title(title + " | pseudo y vs model p_hat")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

def plot_p3_fusion_weights(weights, qualities, title, save_path):
    items = sorted(weights.items(), key=lambda x: -x[1])
    labels = [k for k, _ in items]
    vals = [v for _, v in items]
    qs = [qualities.get(k, 0.0) for k in labels]

    plt.figure(figsize=(10, 4))
    x = np.arange(len(labels))
    plt.bar(x, vals)
    plt.xticks(x, labels, rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.title(title + " | fusion weights (bar)")
    plt.grid(axis="y", alpha=0.25)

    for i, (w, q) in enumerate(zip(vals, qs)):
        plt.text(i, w + 0.01, f"q={q:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

def plot_p4_pair_box(pair_stats, title, save_path):
    labels = ["SS", "TT", "ST", "TS"]
    data = [pair_stats.get(k, np.array([])) for k in labels]

    plt.figure(figsize=(10, 4))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.title(title + f" | pair separation (delta={CONFIG['pair_delta']})")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

def plot_p5_activity_summary(act_to_y_windows, cfg, save_path):
    q = cfg["pair_q"]
    acts = sorted(act_to_y_windows.keys())
    means, stds, topm, botm = [], [], [], []

    for act in acts:
        Ys = np.concatenate(act_to_y_windows[act], axis=0)
        means.append(float(Ys.mean()))
        stds.append(float(Ys.std()))
        lo = float(np.quantile(Ys, q))
        hi = float(np.quantile(Ys, 1.0 - q))
        botm.append(float(Ys[Ys <= lo].mean()))
        topm.append(float(Ys[Ys >= hi].mean()))

    x = np.arange(len(acts))
    plt.figure(figsize=(12, 5))
    plt.errorbar(x, means, yerr=stds, fmt="o", capsize=4, label="mean ± std")
    plt.plot(x, topm, marker="^", linestyle="none", label=f"top {int(q*100)}% mean")
    plt.plot(x, botm, marker="v", linestyle="none", label=f"bottom {int(q*100)}% mean")

    plt.xticks(x, [f"act{a}" for a in acts])
    plt.ylim(0, 1)
    plt.title("Activity summary of pseudo y (transition)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# =========================================================
# 8) Main (LOSO per activity) + representative selection + plots
# =========================================================
def pick_representative_fold(df_act: pd.DataFrame):
    if len(df_act) == 0:
        return None
    med = df_act["mae_soft"].median()
    idx = (df_act["mae_soft"] - med).abs().idxmin()
    return df_act.loc[idx].to_dict()


# =========================================================
# 9) Sweep objective (simple, tunable-free)
# =========================================================
def compute_y_shape_metrics(act_to_y_windows, cfg):
    """
    목적: '중간값 몰림' 완화 + 'top/bottom gap' 증가
    """
    q = cfg["pair_q"]
    acts = sorted(act_to_y_windows.keys())
    rows = []
    for act in acts:
        if act not in act_to_y_windows or len(act_to_y_windows[act]) == 0:
            continue
        Ys = np.concatenate(act_to_y_windows[act], axis=0)
        mean = float(Ys.mean())
        std = float(Ys.std())
        lo = float(np.quantile(Ys, q))
        hi = float(np.quantile(Ys, 1.0 - q))
        botm = float(Ys[Ys <= lo].mean()) if np.any(Ys <= lo) else float("nan")
        topm = float(Ys[Ys >= hi].mean()) if np.any(Ys >= hi) else float("nan")
        gap = float(topm - botm) if (np.isfinite(topm) and np.isfinite(botm)) else float("nan")
        rows.append({"act": act, "y_mean": mean, "y_std": std, "y_botm": botm, "y_topm": topm, "y_gap": gap})

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return {"gap_mean": np.nan, "std_mean": np.nan, "mean_mean": np.nan, "df": df}

    return {
        "gap_mean": float(df["y_gap"].mean()),
        "std_mean": float(df["y_std"].mean()),
        "mean_mean": float(df["y_mean"].mean()),
        "df": df,
    }

def objective_score(df_res, y_metrics):
    """
    ✅ 계층 sweep에서 best를 고르는 스코어
    - y_gap 크게 (전이/정상 대비)
    - y_std 적당히 (몰림 완화)
    - mae_soft 너무 나빠지면 페널티
    """
    mae_mean = float(df_res["mae_soft"].mean()) if len(df_res) else 1e9
    gap = float(y_metrics.get("gap_mean", np.nan))
    std = float(y_metrics.get("std_mean", np.nan))
    if not np.isfinite(gap): gap = 0.0
    if not np.isfinite(std): std = 0.0
    # 가중치는 "튜닝 없이" 고정
    score = (1.0 * gap) + (0.10 * std) - (0.50 * mae_mean)
    return float(score), float(mae_mean), float(gap), float(std)


# =========================================================
# 10) One experiment run (same pipeline, different cfg + out_dir)
# =========================================================
def run_one_experiment(cfg, tag: str):
    # isolate outputs
    out_dir = os.path.join(cfg["out_dir"], tag)
    os.makedirs(out_dir, exist_ok=True)

    # reset seed per run
    set_seed(cfg["seed"])

    print("\n" + "=" * 80)
    print(f"[RUN] {tag}")
    print(f"  quality_tau={cfg['quality_tau']} | ref_q={cfg['sigmoid_ref_q']} | sigmoid_tau={cfg['sigmoid_tau']}")
    print("=" * 80)

    df_all = load_mhealth_df(cfg["data_dir"], cfg["target_activities"])
    blocks_all = create_blocks_by_subject_activity(df_all, cfg["window_size"])
    acts = sorted({b["act"] for b in blocks_all})

    records = []
    act_to_y_windows = {act: [] for act in acts}
    rep_artifacts = {}

    for act in acts:
        print("\n" + "=" * 70)
        print(f"[Activity {act}] Single-activity LOSO")
        print("=" * 70)

        blocks = [b for b in blocks_all if b["act"] == act]
        subjects = sorted({b["subject"] for b in blocks})
        print(f"[Blocks] act={act} total={len(blocks)} | folds={len(subjects)}")

        for test_sub in subjects:
            train_blocks = [b for b in blocks if b["subject"] != test_sub]
            test_blocks  = [b for b in blocks if b["subject"] == test_sub]
            if len(train_blocks) == 0 or len(test_blocks) == 0:
                continue

            train_ds = MainEngineDataset(train_blocks, cfg)
            test_ds  = MainEngineDataset(test_blocks, cfg)
            if len(train_ds) == 0 or len(test_ds) == 0:
                continue

            train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True)
            test_loader  = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False)

            x0, s0, y0 = train_ds[0]
            input_ch = x0.shape[0]

            model = MainEngineNet(
                input_ch=input_ch,
                hidden_dim=cfg["hidden_dim"],
                latent_dim=cfg["latent_dim"],
            ).to(DEVICE)

            train_loss = train_one_fold(model, train_loader, cfg)
            mae_soft, mse_soft = eval_soft_mae_mse(model, test_loader)

            act_to_y_windows[act].extend([d["y"] for d in train_ds.samples])

            records.append({
                "act": act,
                "test_sub": test_sub,
                "train_loss": train_loss["loss"],
                "train_recon": train_loss["recon"],
                "train_soft": train_loss["soft"],
                "train_pair": train_loss["pair"],
                "train_inert": train_loss["inert"],
                "mae_soft": mae_soft,
                "mse_soft": mse_soft,
                "input_ch": input_ch,
            })

            print(f"  Fold test_sub={test_sub:2d} | MAE={mae_soft:.4f} MSE={mse_soft:.4f}")

            b0 = test_blocks[0]
            rep_artifacts.setdefault(act, [])
            rep_artifacts[act].append({
                "test_sub": test_sub,
                "mae_soft": mae_soft,
                "mse_soft": mse_soft,
                "raw0": b0["raw"],
                "subject": b0["subject"],
                "act": b0["act"],
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "input_ch": input_ch,
            })

    df_res = pd.DataFrame(records)
    csv_path = os.path.join(out_dir, "results_loso.csv")
    df_res.to_csv(csv_path, index=False)
    print("\n[Saved]", csv_path)

    # Representative plots per act
    for act in acts:
        df_act = df_res[df_res["act"] == act].reset_index(drop=True)
        if len(df_act) == 0 or act not in rep_artifacts:
            continue

        rep = pick_representative_fold(df_act)
        if rep is None:
            continue
        rep_sub = int(rep["test_sub"])

        cand = None
        for item in rep_artifacts[act]:
            if int(item["test_sub"]) == rep_sub:
                cand = item
                break
        if cand is None:
            continue

        model = MainEngineNet(
            input_ch=int(cand["input_ch"]),
            hidden_dim=cfg["hidden_dim"],
            latent_dim=cfg["latent_dim"],
        ).to(DEVICE)
        model.load_state_dict(cand["model_state"], strict=True)
        model.eval()

        raw0 = cand["raw0"]
        subj = cand["subject"]

        s_fused, y_blk, w_blk, q_blk = detector_fused_on_block(raw0, cfg)
        p_blk = infer_p_hat_on_block(model, raw0, cfg)
        pair_stats = compute_pair_separation_stats(model, raw0, cfg)

        p1_path = os.path.join(out_dir, f"act{act}_P1_raw_and_y_sub{subj}.png")
        plot_p1_raw_and_y(
            raw_24=raw0,
            y=y_blk,
            cfg=cfg,
            title=f"Act{act} RepFold (test_sub={rep_sub})",
            save_path=p1_path
        )

        p2_path = os.path.join(out_dir, f"act{act}_P2_y_vs_phat_sub{subj}.png")
        plot_p2_y_vs_phat(
            y=y_blk,
            p_hat=p_blk,
            cfg=cfg,
            title=f"Act{act} RepFold (test_sub={rep_sub})",
            save_path=p2_path
        )

        p3_path = os.path.join(out_dir, f"act{act}_P3_fusion_weights_sub{subj}.png")
        plot_p3_fusion_weights(
            weights=w_blk,
            qualities=q_blk,
            title=f"Act{act} RepFold (test_sub={rep_sub})",
            save_path=p3_path
        )

        if pair_stats is not None:
            p4_path = os.path.join(out_dir, f"act{act}_P4_pair_separation_sub{subj}.png")
            plot_p4_pair_box(
                pair_stats=pair_stats,
                title=f"Act{act} RepFold (test_sub={rep_sub})",
                save_path=p4_path
            )

        print(f"[Saved plots] act={act} rep_sub={rep_sub} -> {out_dir}")

    p5_path = os.path.join(out_dir, "P5_activity_summary.png")
    plot_p5_activity_summary(act_to_y_windows, cfg, p5_path)
    print("[Saved]", p5_path)

    # metrics for sweep
    y_metrics = compute_y_shape_metrics(act_to_y_windows, cfg)
    score, mae_mean, gap_mean, std_mean = objective_score(df_res, y_metrics)

    # also save y_metrics per act
    y_csv = os.path.join(out_dir, "y_shape_per_act.csv")
    y_metrics["df"].to_csv(y_csv, index=False)
    print("[Saved]", y_csv)

    print("\n[RUN SUMMARY]")
    print(f"  score={score:.6f} | mae_mean={mae_mean:.6f} | gap_mean={gap_mean:.6f} | std_mean={std_mean:.6f}")
    print("Done.\n")

    return {
        "tag": tag,
        "quality_tau": float(cfg["quality_tau"]),
        "sigmoid_ref_q": int(cfg["sigmoid_ref_q"]),
        "sigmoid_tau": float(cfg["sigmoid_tau"]),
        "score": float(score),
        "mae_mean": float(mae_mean),
        "gap_mean": float(gap_mean),
        "std_mean": float(std_mean),
        "out_dir": out_dir,
    }


# =========================================================
# 11) 3-stage hierarchical sweep
# =========================================================
def hierarchical_sweep(base_cfg):
    os.makedirs(base_cfg["out_dir"], exist_ok=True)

    # candidates (고정)
    QUALITY_TAU_CANDS = [1.0, 0.5, 0.25]
    REFQ_CANDS = [80, 90, 95]
    SIGTAU_CANDS = [1.0, 0.7, 0.5]

    sweep_rows = []

    # -----------------------
    # Stage 1: quality_tau
    # -----------------------
    print("\n" + "#" * 90)
    print("[STAGE 1] Sweep quality_tau")
    print("#" * 90)

    best1 = None
    for qt in QUALITY_TAU_CANDS:
        cfg = dict(base_cfg)
        cfg["quality_tau"] = float(qt)
        tag = f"stage1_qualityTau_{qt}"
        row = run_one_experiment(cfg, tag)
        row["stage"] = 1
        sweep_rows.append(row)
        if (best1 is None) or (row["score"] > best1["score"]):
            best1 = row

    print("\n[STAGE 1 BEST]")
    print(best1)

    # -----------------------
    # Stage 2: ref_q (best quality_tau fixed)
    # -----------------------
    print("\n" + "#" * 90)
    print("[STAGE 2] Sweep sigmoid_ref_q (fix best quality_tau)")
    print("#" * 90)

    best2 = None
    for rq in REFQ_CANDS:
        cfg = dict(base_cfg)
        cfg["quality_tau"] = float(best1["quality_tau"])
        cfg["sigmoid_ref_q"] = int(rq)
        tag = f"stage2_refQ_{rq}_qTau_{cfg['quality_tau']}"
        row = run_one_experiment(cfg, tag)
        row["stage"] = 2
        sweep_rows.append(row)
        if (best2 is None) or (row["score"] > best2["score"]):
            best2 = row

    print("\n[STAGE 2 BEST]")
    print(best2)

    # -----------------------
    # Stage 3: sigmoid_tau (best quality_tau + ref_q fixed)
    # -----------------------
    print("\n" + "#" * 90)
    print("[STAGE 3] Sweep sigmoid_tau (fix best quality_tau + ref_q)")
    print("#" * 90)

    best3 = None
    for stau in SIGTAU_CANDS:
        cfg = dict(base_cfg)
        cfg["quality_tau"] = float(best2["quality_tau"])
        cfg["sigmoid_ref_q"] = int(best2["sigmoid_ref_q"])
        cfg["sigmoid_tau"] = float(stau)
        tag = f"stage3_sigTau_{stau}_refQ_{cfg['sigmoid_ref_q']}_qTau_{cfg['quality_tau']}"
        row = run_one_experiment(cfg, tag)
        row["stage"] = 3
        sweep_rows.append(row)
        if (best3 is None) or (row["score"] > best3["score"]):
            best3 = row

    print("\n[STAGE 3 BEST] (FINAL BEST)")
    print(best3)

    # Save sweep summary
    df_sweep = pd.DataFrame(sweep_rows)
    sweep_csv = os.path.join(base_cfg["out_dir"], "sweep_summary.csv")
    df_sweep.to_csv(sweep_csv, index=False)
    print("\n[Saved]", sweep_csv)

    # Save final best pointer
    best_txt = os.path.join(base_cfg["out_dir"], "best_config.txt")
    with open(best_txt, "w") as f:
        f.write("FINAL BEST (Stage3 Best)\n")
        for k in ["quality_tau", "sigmoid_ref_q", "sigmoid_tau", "score", "mae_mean", "gap_mean", "std_mean", "out_dir"]:
            f.write(f"{k}: {best3.get(k)}\n")
    print("[Saved]", best_txt)

    return best3


# =========================================================
# 12) Entry
# =========================================================
def main():
    # out_dir base
    os.makedirs(CONFIG["out_dir"], exist_ok=True)

    print("=" * 70)
    print("Main Engine (CLEAN) - MHEALTH LOSO (per-activity)")
    print("✅ Hierarchical Sweep Enabled")
    print("=" * 70)

    hierarchical_sweep(CONFIG)


if __name__ == "__main__":
    main()


Main Engine (CLEAN) - MHEALTH LOSO (per-activity)
✅ Hierarchical Sweep Enabled

##########################################################################################
[STAGE 1] Sweep quality_tau
##########################################################################################

[RUN] stage1_qualityTau_1.0
  quality_tau=1.0 | ref_q=80 | sigmoid_tau=1.0
[Data] Total samples: 68098 | Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | Acts: [np.int64(6), np.int64(7), np.int64(12)]

[Activity 6] Single-activity LOSO
[Blocks] act=6 total=10 | folds=10
  Fold test_sub= 1 | MAE=0.0960 MSE=0.0136
  Fold test_sub= 2 | MAE=0.0865 MSE=0.0111
  Fold test_sub= 3 | MAE=0.0715 MSE=0.0081
  Fold test_sub= 4 | MAE=0.0785 MSE=0.0093
  Fold test_sub= 5 | MAE=0.0901 MSE=0.0118
  Fold test_sub= 6 | MAE=0.0919 MSE=0.0122
  Fold test_sub= 7 | MAE=0.0708 MSE=0.0089
  Fold test_sub= 8 | MAE=0.0872 MSE=0.0111

/tmp/ipython-input-262303437.py:759: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plots] act=6 rep_sub=2 -> ./out_main_engine/stage1_qualityTau_1.0


/tmp/ipython-input-262303437.py:759: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plots] act=7 rep_sub=1 -> ./out_main_engine/stage1_qualityTau_1.0


...

[Saved plots] act=6 rep_sub=2 -> ./out_main_engine/stage3_sigTau_0.5_refQ_80_qTau_0.25


/tmp/ipython-input-262303437.py:759: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plots] act=7 rep_sub=3 -> ./out_main_engine/stage3_sigTau_0.5_refQ_80_qTau_0.25


/tmp/ipython-input-262303437.py:759: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plots] act=12 rep_sub=4 -> ./out_main_engine/stage3_sigTau_0.5_refQ_80_qTau_0.25
[Saved] ./out_main_engine/stage3_sigTau_0.5_refQ_80_qTau_0.25/P5_activity_summary.png
[Saved] ./out_main_engine/stage3_sigTau_0.5_refQ_80_qTau_0.25/y_shape_per_act.csv

[RUN SUMMARY]
  score=0.271416 | mae_mean=0.087944 | gap_mean=0.304384 | std_mean=0.110043
Done.


[STAGE 3 BEST] (FINAL BEST)
{'tag': 'stage3_sigTau_0.5_refQ_80_qTau_0.25', 'quality_tau': 0.25, 'sigmoid_ref_q': 80, 'sigmoid_tau': 0.5, 'score': 0.2714159221077959, 'mae_mean': 0.08794413544237614, 'gap_mean': 0.30438365042209625, 'std_mean': 0.1100433940688769, 'out_dir': './out_main_engine/stage3_sigTau_0.5_refQ_80_qTau_0.25', 'stage': 3}

[Saved] ./out_main_engine/sweep_summary.csv
[Saved] ./out_main_engine/best_config.txt
